# Safe Pilot App — Scenario 1: Real-Time Weather Alert (Parked Car)

---

### What this notebook does

| Step | Description |
|---|---|
| 1 | Simulates current time as **11:00 AM** with the car **parked** |
| 2 | Connects to **Open-Meteo** (free, no API key needed) to fetch the hourly weather forecast |
| 3 | Injects a **hail event at 12:00 PM** into the forecast (demo scenario) |
| 4 | Evaluates the next 60-minute window for severe weather |
| 5 | Fires a **real-time alert** — console banner + styled HTML panel + browser popup |

> **API used:** [Open-Meteo](https://open-meteo.com/) — completely free, no sign-up, no API key required.
>
> **Why inject hail?** We cannot guarantee real hail will be forecast at your demo location and time. The injection lets us reliably demonstrate the full alert pipeline while still making a live API call for all other weather data.

---
## Cell 1 — Install & Imports

- `requests` — HTTP client for calling the Open-Meteo REST API  
- `IPython.display` — renders styled HTML alerts and triggers browser popups inside Colab  
- `datetime` / `time` — used to simulate the 11 AM clock and compute the 1-hour alert window

In [ ]:
# requests is pre-installed in Colab; the pip call is a safe no-op if already present
!pip install -q requests

import requests
from datetime import datetime, date, timedelta
from time import sleep
from IPython.display import display, HTML, Javascript

print("✅ All imports ready.")

---
## Cell 2 — Configuration

Set the **parked vehicle details** and the **simulated current time** here.

| Parameter | Value | Notes |
|---|---|---|
| `LATITUDE / LONGITUDE` | 13.0827, 80.2707 | Anna Nagar, Chennai (change to your location) |
| `SIMULATED_HOUR` | 11 | Represents 11:00 AM |
| `ALERT_WINDOW_MINUTES` | 60 | Look ahead 60 min for incoming severe weather |
| `HAIL_WMO_CODES` | 96, 99 | WMO standard codes for thunderstorm + hail |

**WMO weather codes for hail:**
- `96` — Thunderstorm with slight hail
- `99` — Thunderstorm with heavy hail

In [ ]:
# ── Parked vehicle ────────────────────────────────────────────────────────────
VEHICLE = {
    "id":      "TN-01-AB-1234",
    "status":  "parked",
    "speed":   0,
    "address": "Anna Nagar, Chennai",
}

# ── Location (Open-Meteo uses lat/lon) ────────────────────────────────────────
LATITUDE  = 13.0827
LONGITUDE = 80.2707

# ── Simulated clock: 11:00 AM today ──────────────────────────────────────────
SIMULATED_HOUR        = 11
ALERT_WINDOW_MINUTES  = 60     # look ahead 60 minutes

simulated_now     = datetime.combine(date.today(), datetime.min.time().replace(hour=SIMULATED_HOUR))
alert_window_end  = simulated_now + timedelta(minutes=ALERT_WINDOW_MINUTES)

# ── WMO codes that represent hail ─────────────────────────────────────────────
HAIL_WMO_CODES    = {96, 99}   # 96=slight hail, 99=heavy hail

# ── Open-Meteo API endpoint ───────────────────────────────────────────────────
OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"

print(f"✅ Config ready.")
print(f"   Vehicle  : {VEHICLE['id']} | {VEHICLE['address']}")
print(f"   Location : lat={LATITUDE}, lon={LONGITUDE}")
print(f"   Simulated time : {simulated_now.strftime('%Y-%m-%d %H:%M')}")
print(f"   Alert window   : {simulated_now.strftime('%H:%M')} → {alert_window_end.strftime('%H:%M')}")

---
## Cell 3 — Fetch Weather Forecast from Open-Meteo API

Calls the Open-Meteo `/v1/forecast` endpoint and retrieves an **hourly forecast** for today.

Fields requested:
| Field | Meaning |
|---|---|
| `weather_code` | WMO weather interpretation code (0=clear … 99=heavy hail) |
| `precipitation` | Total precipitation in mm/hr |
| `precipitation_probability` | Probability of precipitation (%) |
| `wind_speed_10m` | Wind speed at 10 m height (km/h) |
| `temperature_2m` | Air temperature at 2 m (°C) |

In [ ]:
def fetch_forecast(lat: float, lon: float) -> dict:
    """Call Open-Meteo and return the parsed JSON response."""
    params = {
        "latitude":   lat,
        "longitude":  lon,
        "hourly":     "weather_code,precipitation,precipitation_probability,wind_speed_10m,temperature_2m",
        "timezone":   "auto",
        "forecast_days": 1,
    }
    response = requests.get(OPEN_METEO_URL, params=params, timeout=10)
    response.raise_for_status()
    return response.json()


def parse_hourly(api_response: dict) -> list[dict]:
    """Flatten the hourly arrays into a list of per-hour dicts."""
    h = api_response["hourly"]
    return [
        {
            "time":                     datetime.fromisoformat(h["time"][i]),
            "weather_code":             h["weather_code"][i],
            "precipitation_mm":         h["precipitation"][i],
            "precipitation_probability":h["precipitation_probability"][i],
            "wind_speed_kmh":           h["wind_speed_10m"][i],
            "temperature_c":            h["temperature_2m"][i],
        }
        for i in range(len(h["time"]))
    ]


print("⏳ Calling Open-Meteo API ...")
raw_forecast  = fetch_forecast(LATITUDE, LONGITUDE)
hourly_data   = parse_hourly(raw_forecast)

print(f"✅ API call successful — {len(hourly_data)} hourly slots received.")
print(f"   Timezone reported by API : {raw_forecast.get('timezone', 'N/A')}")
print()
print("   Hourly snapshot (first 6 hours):")
print(f"   {'Time':<8} {'WMO Code':<10} {'Precip mm':<12} {'Precip %':<10} {'Wind km/h':<12} {'Temp °C'}")
print("   " + "-" * 62)
for row in hourly_data[:6]:
    print(f"   {row['time'].strftime('%H:%M'):<8} "
          f"{row['weather_code']:<10} "
          f"{row['precipitation_mm']:<12} "
          f"{row['precipitation_probability']:<10} "
          f"{row['wind_speed_kmh']:<12} "
          f"{row['temperature_c']}")

---
## Cell 4 — Inject Simulated Hail at 12:00 PM

Since we cannot guarantee real hail will appear in today's live forecast, this cell **overrides the 12:00 PM slot** with hail data (WMO code `96`) to reliably demonstrate the alert pipeline.

> In a production integration, this cell would be removed — the real API data would flow directly into the evaluator.

In [ ]:
# Target slot: 12:00 PM today (1 hour from simulated 11 AM)
hail_time = simulated_now + timedelta(hours=1)   # 12:00 PM

injected = False
for slot in hourly_data:
    if slot["time"].hour == hail_time.hour and slot["time"].date() == hail_time.date():
        slot["weather_code"]              = 96     # WMO: thunderstorm with slight hail
        slot["precipitation_mm"]          = 18.5   # heavy precipitation
        slot["precipitation_probability"] = 95     # near-certain
        slot["wind_speed_kmh"]            = 72.0   # strong gusts
        slot["_injected"]                 = True   # flag for transparency
        injected = True
        print(f"✅ Hail scenario injected at {slot['time'].strftime('%H:%M')}")
        print(f"   WMO code: 96 (thunderstorm with slight hail)")
        print(f"   Precipitation: {slot['precipitation_mm']} mm | "
              f"Probability: {slot['precipitation_probability']}% | "
              f"Wind: {slot['wind_speed_kmh']} km/h")
        break

if not injected:
    print("⚠️  Could not find the 12:00 PM slot in the forecast. Check timezone.")

---
## Cell 5 — Alert Evaluation Logic

Scans every hourly slot **within the alert window** (`11:00 AM → 12:00 PM`) and checks:

| Check | Condition |
|---|---|
| Hail code present | `weather_code` ∈ `{96, 99}` |
| High precipitation probability | `precipitation_probability` ≥ 70 % |
| Slot is within window | `simulated_now ≤ slot.time ≤ alert_window_end` |

Returns a structured `alert_payload` dict consumed by the display cell.

In [ ]:
# WMO code → human label mapping
WMO_LABELS = {
    0: "Clear sky",
    1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Icy fog",
    51: "Light drizzle", 53: "Drizzle", 55: "Heavy drizzle",
    61: "Light rain", 63: "Rain", 65: "Heavy rain",
    71: "Light snow", 73: "Snow", 75: "Heavy snow",
    80: "Rain showers", 81: "Heavy showers", 82: "Violent showers",
    95: "Thunderstorm",
    96: "Thunderstorm with slight hail",
    99: "Thunderstorm with heavy hail",
}


def evaluate_window(hourly: list, now: datetime, window_end: datetime) -> dict:
    """
    Scan the alert window and return an alert payload if hail is detected.

    Returns:
        dict with keys: alert_required, events, worst_slot
    """
    hail_events = []

    for slot in hourly:
        # Only look at slots within [now, window_end]
        if not (now <= slot["time"] <= window_end):
            continue

        is_hail = slot["weather_code"] in HAIL_WMO_CODES
        high_prob = slot["precipitation_probability"] >= 70

        if is_hail and high_prob:
            hail_events.append(slot)

    if not hail_events:
        return {"alert_required": False, "events": [], "worst_slot": None}

    # Pick the slot with the highest precipitation as the 'worst'
    worst = max(hail_events, key=lambda s: s["precipitation_mm"])

    # How many minutes from now until the worst slot
    eta_minutes = int((worst["time"] - now).total_seconds() / 60)

    return {
        "alert_required": True,
        "events":         hail_events,
        "worst_slot":     worst,
        "eta_minutes":    eta_minutes,
        "event_label":    WMO_LABELS.get(worst["weather_code"], "Severe weather"),
    }


alert_payload = evaluate_window(hourly_data, simulated_now, alert_window_end)

if alert_payload["alert_required"]:
    ws = alert_payload["worst_slot"]
    print(f"🚨 Alert condition detected!")
    print(f"   Event       : {alert_payload['event_label']}")
    print(f"   Arrives at  : {ws['time'].strftime('%H:%M')} "
          f"(in {alert_payload['eta_minutes']} minutes)")
    print(f"   Precip      : {ws['precipitation_mm']} mm | "
          f"Probability: {ws['precipitation_probability']}% | "
          f"Wind: {ws['wind_speed_kmh']} km/h")
else:
    print("✅ No hail detected in the next 60 minutes. No alert needed.")

---
## Cell 6 — Fire the Real-Time Alert

When an alert condition is detected, three notification layers fire simultaneously:

| Layer | Method | Description |
|---|---|---|
| **Console** | `print()` | Formatted text banner — visible in cell output |
| **HTML panel** | `IPython.display.HTML` | Colour-coded alert card rendered in the notebook |
| **Browser popup** | `IPython.display.Javascript` | Native browser `alert()` dialog box |

If no alert is needed, a green all-clear panel is shown instead.

In [ ]:
def fire_alert(vehicle: dict, payload: dict, current_time: datetime):
    """Send all three alert layers based on the evaluation payload."""

    # ── 1. CONSOLE BANNER ────────────────────────────────────────────────────
    print("\n" + "=" * 65)
    if payload["alert_required"]:
        ws = payload["worst_slot"]
        eta = payload["eta_minutes"]
        print("  🚨 SAFE PILOT ALERT — HAIL WARNING")
        print("=" * 65)
        print(f"  Time        : {current_time.strftime('%I:%M %p')}")
        print(f"  Vehicle     : {vehicle['id']}")
        print(f"  Location    : {vehicle['address']}")
        print(f"  Hazard      : {payload['event_label']}")
        print(f"  ETA         : {ws['time'].strftime('%I:%M %p')} ({eta} minutes from now)")
        print(f"  Rainfall    : {ws['precipitation_mm']} mm/hr  |  "
              f"Wind: {ws['wind_speed_kmh']} km/h  |  "
              f"Probability: {ws['precipitation_probability']}%")
        print("-" * 65)
        print("  ⚠️  ACTION: Move your vehicle to covered parking")
        print("      (garage, carport, or underpass) before 12:00 PM.")
    else:
        print("  ✅ SAFE PILOT — ALL CLEAR")
        print("=" * 65)
        print(f"  Time        : {current_time.strftime('%I:%M %p')}")
        print(f"  Vehicle     : {vehicle['id']}  |  {vehicle['address']}")
        print("  Status      : No severe weather in the next 60 minutes.")
    print("=" * 65)

    # ── 2. HTML ALERT PANEL ───────────────────────────────────────────────────
    if payload["alert_required"]:
        ws     = payload["worst_slot"]
        eta    = payload["eta_minutes"]
        html   = f"""
        <div style="
            border: 3px solid #c0392b;
            border-radius: 10px;
            background: #fff5f5;
            padding: 20px 28px;
            margin: 14px 0;
            font-family: Arial, sans-serif;
            max-width: 680px;
        ">
          <div style="display:flex; align-items:center; gap:12px; margin-bottom:12px;">
            <span style="font-size:2.2rem;">🚨</span>
            <div>
              <div style="font-size:1.25rem; font-weight:700; color:#c0392b;">
                SAFE PILOT — HAIL WARNING
              </div>
              <div style="font-size:0.85rem; color:#888;">
                Alert issued at {current_time.strftime('%I:%M %p')} &nbsp;|&nbsp;
                {vehicle['id']} &nbsp;|&nbsp; {vehicle['address']}
              </div>
            </div>
          </div>

          <table style="width:100%; border-collapse:collapse; font-size:0.95rem;">
            <tr style="background:#fdecea;">
              <td style="padding:7px 12px; font-weight:600;">Hazard</td>
              <td style="padding:7px 12px;">{payload['event_label']}</td>
            </tr>
            <tr>
              <td style="padding:7px 12px; font-weight:600;">Expected arrival</td>
              <td style="padding:7px 12px;">
                {ws['time'].strftime('%I:%M %p')} &nbsp;
                <span style="color:#c0392b; font-weight:600;">({eta} minutes from now)</span>
              </td>
            </tr>
            <tr style="background:#fdecea;">
              <td style="padding:7px 12px; font-weight:600;">Rainfall</td>
              <td style="padding:7px 12px;">{ws['precipitation_mm']} mm/hr</td>
            </tr>
            <tr>
              <td style="padding:7px 12px; font-weight:600;">Wind speed</td>
              <td style="padding:7px 12px;">{ws['wind_speed_kmh']} km/h</td>
            </tr>
            <tr style="background:#fdecea;">
              <td style="padding:7px 12px; font-weight:600;">Probability</td>
              <td style="padding:7px 12px;">{ws['precipitation_probability']}%</td>
            </tr>
          </table>

          <div style="
            margin-top:16px;
            background:#c0392b;
            color:white;
            padding:12px 16px;
            border-radius:6px;
            font-size:1rem;
            font-weight:600;
          ">
            ⚠️ ACTION REQUIRED: Move your vehicle to a covered parking area
            (garage, carport, or underpass) before {ws['time'].strftime('%I:%M %p')}.
          </div>
        </div>
        """
    else:
        html = """
        <div style="
            border: 2px solid #27ae60;
            border-radius: 10px;
            background: #f0fff4;
            padding: 18px 24px;
            font-family: Arial, sans-serif;
            max-width: 500px;
        ">
          <span style="font-size:1.5rem;">✅</span>
          <strong style="color:#27ae60; font-size:1.1rem;"> SAFE PILOT — All Clear</strong>
          <p style="margin:8px 0 0; color:#555;">No severe weather detected in the next 60 minutes.
          Your vehicle is safe where it is parked.</p>
        </div>
        """

    display(HTML(html))

    # ── 3. BROWSER POPUP ─────────────────────────────────────────────────────
    if payload["alert_required"]:
        ws  = payload["worst_slot"]
        eta = payload["eta_minutes"]
        popup_msg = (
            f"🚨 SAFE PILOT ALERT\n\n"
            f"HAIL WARNING for {vehicle['address']}\n"
            f"Hail storm arriving at {ws['time'].strftime('%I:%M %p')} "
            f"({eta} min from now).\n\n"
            f"Move your car to covered parking immediately!"
        )
        # Escape single quotes so the JS string is valid
        popup_msg_escaped = popup_msg.replace("'", "\\'")
        display(Javascript(f"alert('{popup_msg_escaped}')"))


# ── RUN THE ALERT ─────────────────────────────────────────────────────────────
fire_alert(VEHICLE, alert_payload, simulated_now)

---
## Cell 7 — Full Hourly Forecast Table (Reference)

Displays the complete 24-hour forecast so you can see all the raw API data alongside the injected hail slot.

In [ ]:
print(f"\n{'─'*78}")
print(f"  Full 24-Hour Forecast — {VEHICLE['address']}")
print(f"{'─'*78}")
print(f"  {'Time':<8} {'WMO':<5} {'Weather Description':<36} {'Precip mm':<11} {'Prob %':<8} {'Wind'}")
print(f"  {'─'*74}")

for slot in hourly_data:
    label    = WMO_LABELS.get(slot["weather_code"], f"Code {slot['weather_code']}")
    injected = " ← DEMO HAIL" if slot.get("_injected") else ""
    # Highlight the alert window
    in_window = simulated_now <= slot["time"] <= alert_window_end
    marker    = ">>" if in_window else "  "
    print(f"{marker} {slot['time'].strftime('%H:%M'):<8} "
          f"{slot['weather_code']:<5} "
          f"{label:<36} "
          f"{slot['precipitation_mm']:<11} "
          f"{slot['precipitation_probability']:<8} "
          f"{slot['wind_speed_kmh']} km/h"
          f"{injected}")

print(f"{'─'*78}")
print("  >> = inside the 60-minute alert window")